# Homework 11: LangChain Pipelines, RAG, and LoRA

---

### Objectives

In this assignment you will practice three key concepts from Lecture 11:

1. **LangChain Pipelines** -- Build a multi-step LLM chain that translates, summarizes, detects language, calls Python functions, and generates a follow-up message.
2. **Retrieval-Augmented Generation (RAG)** -- Build a complete RAG pipeline: load a PDF, chunk text, embed into a vector database, retrieve relevant chunks, and answer questions.
3. **LoRA (Low-Rank Adaptation)** -- Implement the LoRA math from scratch to understand how parameter-efficient fine-tuning works.

---

### Skills You Will Practice

- Using LangChain prompt templates, LLMChain, and SequentialChain
- Defining function schemas for LLM tool use
- Building a RAG pipeline with ChromaDB and sentence embeddings
- Implementing low-rank matrix decomposition for efficient fine-tuning
- Computing parameter savings with different LoRA ranks

---

### Guidelines

- **Do not modify** the structure or logic of the provided code -- your task is to understand and complete it.
- Problems 1-2 require Azure OpenAI API access (your Foundry endpoint).
- Problem 3 runs on CPU with PyTorch only.
- Total: **25 points**

---

### Setup

In [6]:
import subprocess, sys

def install_if_missing(package):
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--quiet"])

install_if_missing("torch")
install_if_missing("numpy")
install_if_missing("openai")
install_if_missing("langchain")
install_if_missing("langchain_community")
install_if_missing("pypdf")
install_if_missing("sentence_transformers")
install_if_missing("chromadb")

import warnings
warnings.filterwarnings("ignore")

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
import json

torch.manual_seed(42)

In [8]:
# Azure OpenAI Client Setup
! pip install openai azure-identity --quiet

from azure.identity import DeviceCodeCredential, get_bearer_token_provider
from openai import AzureOpenAI

credential = DeviceCodeCredential()
token_provider = get_bearer_token_provider(
    credential,
    "https://cognitiveservices.azure.com/.default"
)

client = AzureOpenAI(
    azure_endpoint="https://mlearn530nusmith-aiservices.services.ai.azure.com/",
    azure_ad_token_provider=token_provider,
    api_version="2024-02-01"
)

try:
    print("Testing connection...")
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": "Hello! Confirm connection."}]
    )
    print("[SUCCESS]", response.choices[0].message.content)
except Exception as e:
    print(f"[FAILURE] {e}")


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Testing connection...
To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code EY77QTQ4J to authenticate.
[SUCCESS] Hello! Connection confirmed—I'm here and ready to assist. How can I help you today? 😊


## Problem 1: LangChain Multi-Step Pipeline (40 points)

### Background

In this problem you will build a multi-step Chain-of-Thought pipeline using LangChain that:
1. Translates a customer review to English
2. Summarizes it in one sentence
3. Detects the original language
4. Calls Python functions to suggest a product and compute a confidence score
5. Generates a follow-up response in the original language

### Recap from lecture

- **SequentialChain** chains multiple LLMChains together, passing outputs as inputs
- **Function schemas** let the LLM know about available tools
- **ChatPromptTemplate** structures prompts with variables

In [20]:
# LangChain setup with Azure (uses the same credential from above)
! pip install langchain langchain-openai --quiet

from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser   
from langchain_core.runnables import RunnablePassthrough    
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Reuse the same token provider from the client setup cell
llm = AzureChatOpenAI(
    azure_deployment="gpt-4o",
    api_version="2024-02-01",
    azure_endpoint="https://mlearn530nusmith-aiservices.services.ai.azure.com/",
    azure_ad_token_provider=token_provider,
)
print("LangChain LLM initialized.")


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
LangChain LLM initialized.


In [21]:
# Python helper functions (provided -- do not modify)

def suggest_product(summary):
    """Suggest a product category based on keywords in the review summary."""
    if "battery" in summary.lower():
        return "Portable Power Bank"
    elif "keyboard" in summary.lower():
        return "Mechanical Keyboard"
    elif "camera" in summary.lower():
        return "Webcam"
    else:
        return "Gift Card"

def confidence_score(summary):
    """Simulates a classifier confidence score."""
    if any(x in summary.lower() for x in ["battery", "keyboard", "camera"]):
        return 0.9
    return 0.6

print("Helper functions loaded.")

Helper functions loaded.


In [22]:
# TODO: Define the prompt templates and chains (30 points)
#
# You need to create 3 LLMChains and combine them with SequentialChain:
#
# Chain 1: Translate Review to English
#   - Prompt: "Translate this customer review to English:\n\n{Review}"
#   - output_key: "English_Review"
#   Hint: first_prompt = ChatPromptTemplate.from_template("...")
#         chain_one = LLMChain(llm=llm, prompt=first_prompt, output_key="English_Review")
#
# Chain 2: Summarize the English Review
#   - Prompt: "Summarize this review in one sentence:\n\n{English_Review}"
#   - output_key: "summary"
#
# Chain 3: Detect Language of original review
#   - Prompt: "Identify the language of this review:\n\n{Review}"
#   - output_key: "language"
#
# Then combine with SequentialChain:
#   sequential_chain = SequentialChain(
#       chains=[chain_one, chain_two, chain_three],
#       input_variables=["Review"],
#       output_variables=["English_Review", "summary", "language"],
#       verbose=True
#   )

# TODO: Your code here

first_prompt = ChatPromptTemplate.from_template("Translate this customer review to English:\n\n{Review}")
chain_one = first_prompt | llm | StrOutputParser()

second_prompt = ChatPromptTemplate.from_template("Summarize this review in one sentence:\n\n{English_Review}")
chain_two = second_prompt | llm | StrOutputParser()

third_prompt = ChatPromptTemplate.from_template("Identify the language of this review:\n\n{Review}")
chain_three = third_prompt | llm | StrOutputParser()

# translate_chain = translate_prompt | llm | StrOutputParser()
# summary_chain = summary_prompt | llm | StrOutputParser()
# language_chain = language_prompt | llm | StrOutputParser()

sequential_chain = (
    RunnablePassthrough
    .assign(
        English_Review=chain_one
    )
    .assign(
        summary=lambda x: chain_two.invoke({
            "English_Review": x["English_Review"]
        })
    )
    .assign(
        language=chain_three
    )
    .assign(
        product=lambda x: suggest_product(x["summary"])
    )
    .assign(
        confidence=lambda x: confidence_score(x["summary"])
    )
)

In [23]:
# TODO: Run the pipeline and generate final response (10 points)
#
# Hints:
# - Run: partial_outputs = sequential_chain({"Review": "J'adore la batterie..."})
# - Extract summary: summary = partial_outputs["summary"]
# - Call suggest_product(summary) and confidence_score(summary)
# - Create a final_prompt that writes a polite response in the detected language
# - Use LLMChain to generate the final message

review_input = {
    "Review": "J'adore la batterie externe, elle se charge tres vite et tient longtemps."
}
final_prompt = ChatPromptTemplate.from_template("Reply politely to the review in the detected language\n\n{Review}")
chain_final = final_prompt | llm | StrOutputParser()


# TODO: Run sequential_chain, call functions, generate final response
partial_outputs = sequential_chain.invoke(review_input)
summary = partial_outputs["summary"]
language = partial_outputs["language"]
product = suggest_product(summary)
confidence = confidence_score(summary)
# ... create final_chain and invoke it ...
final_chain = (
    RunnablePassthrough
    .assign(
        English_Review=chain_one
    )
    .assign(
        summary=lambda x: chain_two.invoke({
            "English_Review": x["English_Review"]
        })
    )
    .assign(
        language=chain_three
    )
    .assign(
        product=lambda x: suggest_product(x["summary"])
    )
    .assign(
        confidence=lambda x: confidence_score(x["summary"])
    )
    .assign(reply=chain_final)
)
result = final_chain.invoke({
    "Review": review_input["Review"],
    "Summary": summary,
    "Language": language,
    "Product": product,
    "Confidence": confidence,
})


# Print results
for k, v in result.items():
    print(f"  {k}: {v}")

  Review: J'adore la batterie externe, elle se charge tres vite et tient longtemps.
  Summary: The external battery is highly efficient, offering quick charging and long-lasting power.
  Language: The language of the review is **French**.
  Product: Portable Power Bank
  Confidence: 0.9
  English_Review: I love the external battery; it charges very quickly and lasts a long time.
  summary: The external battery is highly efficient, with fast charging and long-lasting performance.
  language: The language of this review is **French**.
  product: Portable Power Bank
  confidence: 0.9
  reply: Merci beaucoup pour votre aimable commentaire ! Nous sommes ravis d'apprendre que vous appréciez votre batterie externe et qu'elle répond à vos attentes. Si vous avez besoin d'aide ou d'autres informations, n'hésitez pas à nous contacter. 😊


## Problem 2: Retrieval-Augmented Generation (RAG) with Climate Report (40 points)

### Background

In this problem, you will implement a basic Retrieval-Augmented Generation (RAG) pipeline using a publicly available climate change report (PDF).

### What You Will Do:
- Load and extract text from a real PDF document (climate report)
- Chunk the text at the character-level using `RecursiveCharacterTextSplitter`
- Further chunk into token-level segments using `SentenceTransformersTokenTextSplitter`
- Convert each chunk into a vector embedding using `SentenceTransformerEmbeddingFunction`
- Store these vectors in a vector database using `ChromaDB`
- Retrieve the top chunks for a given query and send them with the query to an LLM
- Use the LLM to answer the question based solely on retrieved context

### Important
Download the IPCC climate report PDF and place it in the same directory as this notebook:
https://www.ipcc.ch/report/ar6/syr/ (use the Longer Report PDF)

In [24]:
# PDF Load
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter, SentenceTransformersTokenTextSplitter
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
reader = PdfReader("IPCC_AR6_SYR_LongerReport.pdf")

pdf_texts = []

for i, page in enumerate(reader.pages):
    text = page.extract_text()
    if text:
        pdf_texts.append(text.strip())

    if i % 10 == 0:
        print(f"Processed page {i}/{len(reader.pages)}", flush=True)

full_text = "\n\n".join(pdf_texts)
print("Finished PDF extraction", flush=True)

Ignoring wrong pointing object 110 0 (offset 0)
Ignoring wrong pointing object 531 0 (offset 0)
Ignoring wrong pointing object 968 0 (offset 0)
Ignoring wrong pointing object 998 0 (offset 0)
Ignoring wrong pointing object 1034 0 (offset 0)
Ignoring wrong pointing object 1221 0 (offset 0)
Ignoring wrong pointing object 1224 0 (offset 0)
Ignoring wrong pointing object 1226 0 (offset 0)
Ignoring wrong pointing object 1228 0 (offset 0)
Ignoring wrong pointing object 1230 0 (offset 0)
Ignoring wrong pointing object 1232 0 (offset 0)
Ignoring wrong pointing object 1233 0 (offset 0)
Ignoring wrong pointing object 1237 0 (offset 0)
Ignoring wrong pointing object 1354 0 (offset 0)
Ignoring wrong pointing object 1380 0 (offset 0)
Ignoring wrong pointing object 1397 0 (offset 0)
Ignoring wrong pointing object 1399 0 (offset 0)
Ignoring wrong pointing object 1402 0 (offset 0)
Ignoring wrong pointing object 1708 0 (offset 0)
Ignoring wrong pointing object 1711 0 (offset 0)
Ignoring wrong pointing 

Processed page 0/81
Processed page 10/81
Processed page 20/81
Processed page 30/81
Processed page 40/81
Processed page 50/81
Processed page 60/81
Processed page 70/81
Processed page 80/81
Finished PDF extraction


In [28]:
# TODO: Build a RAG pipeline (40 points)
#
# Required imports:
import chromadb
from chromadb.utils import embedding_functions
from pypdf import PdfReader
from langchain_text_splitters import (
  RecursiveCharacterTextSplitter,
  SentenceTransformersTokenTextSplitter
)
#
# Step 1: Load the PDF
#   reader = PdfReader("IPCC_AR6_SYR_LongerReport.pdf")
#   pdf_texts = [page.extract_text().strip() for page in reader.pages if page.extract_text()]
#   full_text = "\n\n".join(pdf_texts)
#
# Step 2: Split into character chunks
#   Use RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
#   Example:
#       char_splitter = RecursiveCharacterTextSplitter(
#           chunk_size=800,
#           chunk_overlap=100
#       )
#       char_chunks = char_splitter.split_text(full_text)
#
# Step 3: Split into token chunks
#   Use SentenceTransformersTokenTextSplitter(tokens_per_chunk=128, chunk_overlap=0)
#   Example:
#       token_splitter = SentenceTransformersTokenTextSplitter(
#           tokens_per_chunk=128,
#           chunk_overlap=0
#       )
#       token_chunks = []
#       for chunk in char_chunks:
#           token_chunks.extend(token_splitter.split_text(chunk))
#
# Step 4: Build a ChromaDB vector store
#   Create an embedding model:
#       embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
#           model_name="all-MiniLM-L6-v2"
#       )
#   Create a Chroma client:
#       chroma_client = chromadb.Client()
#   Create a collection:
#       collection = chroma_client.create_collection(
#           name="ipcc_collection",
#           embedding_function=embedding_function
#       )
#   Add all token chunks:
#       collection.add(
#           ids=[str(i) for i in range(len(token_chunks))],
#           documents=token_chunks
#       )
#
# Step 5: Define rag(question)
#   Inside the function:
#   1. Retrieve the top 5 most relevant chunks:
#       results = collection.query(
#           query_texts=[question],
#           n_results=5
#       )
#   2. Combine the retrieved chunks into a single context string:
#       context = "\n\n".join(results["documents"][0])
#   3. Create a prompt containing:
#       - The retrieved context
#       - The user's question
#       - Instructions to answer only using the provided context
#   4. Call:
#       client.chat.completions.create(...)
#       using the same OpenAI client from Question 1.
#   5. Return the generated answer.
#
# Step 6: Run the queries below (do not modify them)
#   Call rag(question) for each question and print the answer.

# TODO: Steps 1-4 here
# Load the PDF
full_text = "\n\n".join(pdf_texts)
# Split into character chunks
char_splitter = RecursiveCharacterTextSplitter(
  chunk_size=800,
  chunk_overlap=100
)
char_chunks = char_splitter.split_text(full_text)
# Split into token chunks
token_splitter = SentenceTransformersTokenTextSplitter(
  tokens_per_chunk=128,
  chunk_overlap=0,
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

token_chunks = []
for chunk in char_chunks:
  token_chunks.extend(token_splitter.split_text(chunk))

# Build a ChromaDB vector store, Create an embedding model
embedding_function = SentenceTransformerEmbeddingFunction(
  model_name="all-MiniLM-L6-v2"
)
chroma_client = chromadb.Client()
chroma_collection = chroma_client.get_or_create_collection(
  name="ipcc_collection",
  embedding_function=embedding_function
)
chroma_collection.add(
  ids=[str(i) for i in range(len(token_chunks))],
  documents=token_chunks
)

# TODO: Step 5 - define rag(query, retrieved_documents, model="gpt-4o"):
def rag(query, retrieved_documents, model="gpt-4o"):
    context = "\n\n".join(retrieved_documents)
    prompt = f"""
    Use only the provided context to answer the question.
    Context: {context}
    Question: {query} """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# Step 6: Run queries (DO NOT MODIFY)
queries = [
    "What are the main causes of global warming mentioned in the report?",
    "What mitigation strategies are discussed?",
    "Does the report include projections for sea level rise?"
]

for query in queries:
    results = chroma_collection.query(query_texts=[query], n_results=5)
    retrieved_docs = results['documents'][0]
    answer = rag(query, retrieved_docs)
    print(f"Query: {query}\nAnswer: {answer}\n{'-'*80}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 18389.81it/s]


Query: What are the main causes of global warming mentioned in the report?
Answer: The report identifies human influence, particularly **human-caused CO₂ emissions**, as the **main driver** of global warming. It highlights human impact as the **main cause** of the global retreat of glaciers, decrease in Arctic sea ice area, reduced Northern Hemisphere spring snow cover, surface melting of the Greenland ice sheet, and the acidification of the surface open ocean. It also emphasizes that current global warming is driven by contributions from all anthropogenic forcers.
--------------------------------------------------------------------------------
Query: What mitigation strategies are discussed?
Answer: The mitigation strategies discussed are:

1. **Addressing Barriers to Feasibility and Effectiveness**: Overcoming limits to feasibility and effectiveness of responses by addressing barriers such as economic and technological challenges.

2. **Enhancing Air Quality Improvement**: Considerin

## Problem 3: Understanding LoRA (Low-Rank Adaptation) (20 points)

### Background

LoRA is a parameter-efficient fine-tuning method. Instead of updating a full weight matrix W (d x d), LoRA learns two small matrices A (r x d) and B (d x r) where r << d. The fine-tuned weight becomes:

**W_FT = W + (alpha / r) * B @ A**

This means:
- W stays frozen (no gradients needed for the original weights)
- Only A and B are trained (much fewer parameters)
- alpha controls the strength of the LoRA update

### Recap from lecture

- Full fine-tuning: 1000x1000 = 1,000,000 parameters to update
- LoRA with r=2: 1000x2 + 2x1000 = 4,000 parameters (250x fewer)
- The weight update matrix delta_W has low rank (most singular values are near zero)

In [29]:
# TODO Part A: Implement LoRALayer (10 points)
#
# Hints:
# - The original layer stays frozen: set requires_grad = False on its parameters
# - A has shape (rank, d_in), initialized with small random values (* 0.01)
# - B has shape (d_out, rank), initialized to zeros
# - forward: original_out + (alpha / rank) * (x @ A.T @ B.T)
#
# The formula from lecture: W_FT = W + (alpha/r) * B @ A

class LoRALayer(nn.Module):
    def __init__(self, original_layer, rank=4, alpha=1.0):
        """
        Initialize a LoRA (Low-Rank Adaptation) layer.
        Instead of fine-tuning all parameters of a neural network layer,
        LoRA keeps the original weights frozen and learns a small low-rank
        update. This greatly reduces the number of trainable parameters.
        
        Parameters
        ----------

        original_layer : nn.Linear
            The pretrained linear layer that we want to adapt.

        rank : int
            The rank (r) of the low-rank decomposition.
            Smaller values use fewer trainable parameters.
            Typical values are 4, 8, 16, or 32.

        alpha : float
            Scaling factor used to control the strength of the LoRA update.
            The final LoRA output is multiplied by (alpha / rank).

        Example
        -------

        Suppose the original layer has:
            input dimension  = 1024
            output dimension = 4096

        A full weight matrix would contain:
            4096 x 1024 = 4,194,304 parameters

        With rank = 8, LoRA learns only:
            A: 8 x 1024      = 8,192 parameters
            B: 4096 x 8      = 32,768 parameters
            Total            = 40,960 parameters

        which is roughly 100x fewer trainable parameters.
        """
        super().__init__()
        d_in = original_layer.in_features
        d_out = original_layer.out_features

        # Freeze the original weights
        self.original = original_layer
        self.original.weight.requires_grad = False
        if self.original.bias is not None:
            self.original.bias.requires_grad = False

        # LoRA matrices: A is (rank x d_in), B is (d_out x rank)
        self.A = nn.Parameter(torch.randn(rank, d_in) * 0.01)
        self.B = nn.Parameter(torch.zeros(d_out, rank))
        self.alpha = alpha
        self.rank = rank
        
    def forward(self, x):
        # TODO:
        # Compute the output of the frozen original layer.
        #
        # Hint:
        #   A PyTorch layer can be called like a function:
        #
        #       output = layer(input)
        #

        # TODO:
        # Compute the LoRA update:
        #
        #       x @ A.T @ B.T
        #
        # where:
        #   @  = matrix multiplication
        #   .T = matrix transpose
        #
        # Shape check:
        #
        #   x       : (batch_size, d_in)
        #   A.T     : (d_in, rank)
        #   B.T     : (rank, d_out)
        #
        # Result:
        #   lora_out: (batch_size, d_out)

        # TODO:
        # Scale the LoRA update by:
        #
        #       alpha / rank
        #
        # and add it to the original output.
        #
        # Return the final result.
        base = self.original_layer(x) 
        lora_update = x @ self.A.T @ self.B.T
        scaled = (self.alpha / self.rank) * lora_update
        return base + scaled


# Test your implementation
base_layer = nn.Linear(512, 512)
lora_layer = LoRALayer(base_layer, rank=4, alpha=1.0)

total_base = sum(p.numel() for p in base_layer.parameters())
total_lora = sum(p.numel() for p in [lora_layer.A, lora_layer.B])
print(f"Base parameters: {total_base:,}")
print(f"LoRA parameters: {total_lora:,}")
print(f"Reduction: {total_base / total_lora:.0f}x")

Base parameters: 262,656
LoRA parameters: 4,096
Reduction: 64x


In [30]:
# TODO Part B: Compare different LoRA ranks (10 points)
#
# Hints:
# - For a d x d matrix, full params = d * d
# - LoRA params with rank r = r*d + d*r = 2*r*d
# - Fill in the table for ranks [1, 2, 4, 8, 16, 32, 64]
# - Also compute SVD of a random delta_W to verify the lecture's claim

print("LoRA Parameter Analysis for a 512x512 weight matrix:")
print(f"{'Rank':>6} | {'LoRA Params':>12} | {'% of Full':>10} | {'Reduction':>10}")
print("-" * 50)

d = 512
full_params = d * d

for rank in [1, 2, 4, 8, 16, 32, 64]:
    # TODO: compute lora_params, pct, reduction
    lora_params = 2 * d * rank  # Replace
    pct = 2 * d * rank / full_params  # Replace
    reduction = 1/pct  # Replace
    print(f"{rank:>6} | {lora_params:>12,} | {pct:>9.2f}% | {reduction:>9.0f}x")

# TODO: Compute SVD of a random 512x512 matrix
delta_W = torch.randn(512, 512) * 0.01
U, S, Vh = torch.linalg.svd(delta_W)
# Print top 5 and bottom 5 singular values
# Print what % of total energy is in the top 4 and top 16 singular values
print("Top 5: ", S[0:5])
print("Bottom 5: ", S[-5:])
total_energy = torch.sum(S**2)
energy_top4 = torch.sum(S[:4]**2)
energy_top16 = torch.sum(S[:16]**2)

print("Pct energy top 4: ", energy_top4/total_energy)
print("Pct energy top 16 5: ",energy_top16/total_energy)

LoRA Parameter Analysis for a 512x512 weight matrix:
  Rank |  LoRA Params |  % of Full |  Reduction
--------------------------------------------------
     1 |        1,024 |      0.00% |       256x
     2 |        2,048 |      0.01% |       128x
     4 |        4,096 |      0.02% |        64x
     8 |        8,192 |      0.03% |        32x
    16 |       16,384 |      0.06% |        16x
    32 |       32,768 |      0.12% |         8x
    64 |       65,536 |      0.25% |         4x
Top 5:  tensor([0.4484, 0.4439, 0.4420, 0.4356, 0.4315])
Bottom 5:  tensor([0.0024, 0.0014, 0.0011, 0.0005, 0.0003])
Pct energy top 4:  tensor(0.0300)
Pct energy top 16 5:  tensor(0.1119)
